In [ ]:
import os
import pandas as pd
import numpy as np

def generate_attribution_dashboard(ledger_path="shadow_ledger_candidates_v4.csv"):
    if not os.path.exists(ledger_path):
        print(f"Missing ledger: {ledger_path}")
        return

    df = pd.read_csv(ledger_path)
    print("\n" + "=" * 80)
    print("V4 PROFIT ATTRIBUTION DASHBOARD")
    print("=" * 80)
    print(f"Ledger: {ledger_path} | rows={len(df)}")

    for col in ["is_trade_live", "is_research_log", "is_rejected", "optimizer_candidate", "is_backfilled"]:
        if col in df.columns:
            print(f"\n{col}:")
            print(df[col].value_counts(dropna=False).to_string())

    print("\nPNL SUMMARY:")
    for col in ["Outcome_PnL", "Provisional_PnL", "Ret_15M", "Ret_30M", "Ret_1H", "Ret_3H"]:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        if s.empty:
            print(f"{col}: no data")
            continue
        gains = s[s > 0].sum()
        losses = -s[s < 0].sum()
        pf = gains / losses if losses > 0 else (99.0 if gains > 0 else 0.0)
        print(f"{col}: n={len(s)} sum={s.sum()*100:.2f}% mean={s.mean()*100:.3f}% win={(s>0).mean()*100:.1f}% PF={pf:.2f}")

    if "optimizer_candidate" in df.columns:
        focus_mask = df["optimizer_candidate"].astype(str).str.lower().isin(["true", "1", "yes"])
        focus = df[focus_mask].copy()
        print(f"\nPROFIT-FOCUS CANDIDATES: {len(focus)}")
        if not focus.empty and "Outcome_PnL" in focus.columns:
            focus["Outcome_PnL"] = pd.to_numeric(focus["Outcome_PnL"], errors="coerce")
            closed = focus.dropna(subset=["Outcome_PnL"])
            if not closed.empty:
                cols = [c for c in ["timestamp_utc", "symbol", "side", "regime", "final_proba", "size_usd", "Outcome_PnL", "Outcome_Status", "Exit_Reason", "profit_focus_reason"] if c in closed.columns]
                print(closed[cols].tail(20).to_string(index=False))

    if {"side", "regime", "Outcome_PnL"}.issubset(df.columns):
        closed = df.dropna(subset=["Outcome_PnL"]).copy()
        closed["Outcome_PnL"] = pd.to_numeric(closed["Outcome_PnL"], errors="coerce")
        print("\nBY SIDE/REGIME FINAL OUTCOME:")
        if not closed.empty:
            report = closed.groupby(["side", "regime"])["Outcome_PnL"].agg(
                Trades="count",
                Total_Pct=lambda x: x.sum() * 100,
                Avg_Bps=lambda x: x.mean() * 10000,
                Win_Rate=lambda x: (x > 0).mean() * 100,
            ).reset_index()
            print(report.to_string(index=False))
    print("=" * 80)

generate_attribution_dashboard()



V4 PROFIT ATTRIBUTION DASHBOARD
Ledger: shadow_ledger_candidates_v4.csv | rows=114

is_trade_live:
is_trade_live
False    114

is_research_log:
is_research_log
True     81
False    33

is_rejected:
is_rejected
False    81
True     33

optimizer_candidate:
optimizer_candidate
False    107
True       7

is_backfilled:
is_backfilled
True     77
False    37

PNL SUMMARY:
Outcome_PnL: n=77 sum=-26.31% mean=-0.342% win=20.8% PF=0.17
Provisional_PnL: n=25 sum=-2.07% mean=-0.083% win=28.0% PF=0.17
Ret_15M: n=94 sum=-22.32% mean=-0.237% win=25.5% PF=0.47
Ret_30M: n=68 sum=-14.53% mean=-0.214% win=22.1% PF=0.45
Ret_1H: n=30 sum=-15.00% mean=-0.500% win=16.7% PF=0.26
Ret_3H: no data

PROFIT-FOCUS CANDIDATES: 7
                   timestamp_utc    symbol  side  regime  final_proba  size_usd  Outcome_PnL Outcome_Status Exit_Reason profit_focus_reason
      2026-05-15 14:15:18.216210  HYPEUSDT SHORT       0     0.647621  4.877000      -0.0053    FINAL_EVENT SL_OR_TRAIL PROFIT_FOCUS_SHORT0
      2026